# mva_06-ロジスティック重回帰分析

---

## **要点まとめ**

**Part Ⅲ：確率の最大化**

線形回帰分析では、「数値データ」から「数値データ」を予測しました。今回学ぶロジスティック回帰分析は、**「数値データ」** から **「質的データ（カテゴリ）」** を予測するの手法です。これまでの「誤差（距離）の最小化」という幾何学的なアプローチから視点を変え、データが発生する **「確率（尤度：もっともらしさ）」** に着目して最適なモデルを探索するアプローチを学びます。

### **1. 回帰から分類へ：確率は予測できるか？**

世の中には「合格/不合格」「購入する/しない」のような **質的データ（2値のカテゴリ）** を予測したい場面が多く存在します。これを **分類（Classification）** と呼びます。

データを行列（表）で整理すると、以下のようになります。
目的変数 $y$ が連続値ではなく、 **$\{0, 1\}$ のカテゴリ（質的データ）** をとる点が重回帰分析との最大の違いです。

| | 目的変数| 説明変数1 | ... | 説明変数d |
|---|:---:|:---:|:---:|:---:|
| **サンプル1** | $y_{1}$ (0/1) | $x_{11}$  | ... | $x_{1d}$ |
| ... | ... | ... | ... | ... |
| **サンプル i** | $y_{i}$ (0/1) | $x_{i1}$ | ... | $x_{id}$ |
| ... | ... | ... | ... | ... |
| **サンプルn** | $y_{n}$ (0/1) | $x_{n1}$ | ... | $x_{nd}$ |

<br>
通常の線形回帰をそのまま分類（0または1の予測）に使うと、以下の問題が発生します。

- **範囲の問題**: 予測値が $1$ を超えたり $0$ を下回ったりするため、「確率」として解釈できない。
- **感度の問題**: 外れ値（極端に遠いデータ）の影響を受けて、境界線が大きくズレてしまう。

<br>

### **2. ロジスティック回帰モデル**

あるサンプル $i$ について、線形結合の結果 $z_i$ を計算し、シグモイド関数 (Sigmoid function) を使って強制的に $0 \sim 1$ の範囲（確率）に変換します。

#### **Step 1: 線形予測値（スコア）の計算**
説明変数 $x_{i1}, \dots, x_{id}$ を重み付きで足し合わせます。

$$z_i = w_0 + w_1 x_{i1} + w_2 x_{i2} + \dots + w_d x_{id} = \mathbf{w}^\top \mathbf{x}_i$$

$z_i$ は $-\infty \sim +\infty$ の値をとります。

#### **Step 2: 確率への変換**
シグモイド関数 $\sigma(z)$ を通して、$0 \sim 1$ の確率 $p_i$ に変換します。

$$p_i = \sigma(z_i) = \frac{1}{1 + e^{-z_i}}$$

$z_i = 0$ のとき確率 $p_i = 0.5$ となり、ここが分類の **決定境界（Decision Boundary）** となります。

$z_i$ が大きくなるほど $p_i$ は $1$ に近づき、$z_i$ が小さくなるほど $p_i$ は $0$ に近づきます。

### **3. 最尤推定法 (Maximum Likelihood Estimation)**

モデルのパラメータ $\mathbf{w}$ を決定するために、最小二乗法（誤差の最小化）ではなく、**最尤法（確率の最大化）** を用います。

「手元にあるデータ $(y_1, y_2, \dots, y_n)$ が観測されたのは、偶然ではなく、起きるべくして起きた（確率が最大になる現象だった）」と考えます。
全データの同時確率（尤度関数 $L$）は以下の積で表されます。

$$L(\mathbf{w}) = \prod_{i=1}^n P(y_i | \mathbf{x}_i; \mathbf{w}) = \prod_{i=1}^n p_i^{y_i} (1-p_i)^{1-y_i}$$

この積 $L(\mathbf{w})$ を最大化することが目的ですが、計算の都合上、以下のように式変形を行います。

#### **Step 1: 積から和へ（対数尤度）**

確率は $0 \sim 1$ の小さい値をとるため、これらを何回も掛け合わせると値が限りなく $0$ に近づき、コンピュータ上で計算できなくなります（アンダーフロー）。
そこで、対数 $\log$ をとることで、掛け算を足し算に変換 します。対数は単調増加関数なので、元の $L$ が最大になる場所と $\log L$ が最大になる場所は同じです。

$$\log L(\mathbf{w}) = \sum_{i=1}^n \left[ y_i \log p_i + (1-y_i) \log(1-p_i) \right]$$

#### **Step 2: 最大化から最小化へ（交差エントロピー誤差）**

一般的に最適化問題は「コスト（誤差）の最小化」として定式化されます。そこで、全体に $-1$ を掛けて符号を反転させ、「負の対数尤度」を最小化する問題 に置き換えます。これを **交差エントロピー誤差 (Cross-Entropy Loss)** と呼びます。

$$J(\mathbf{w}) = - \log L(\mathbf{w}) = - \sum_{i=1}^n \left[ y_i \log p_i + (1-y_i) \log(1-p_i) \right]$$

<br>

### **4. 最適化の実現：勾配降下法 (Gradient Descent)**

最適解の条件である $\nabla J(\mathbf{w}) = \mathbf{0}$ は必ずしも解析的に解くことはできません。目的関数 $J(\mathbf{w})$ の勾配$\nabla J(\mathbf{w})$の情報を用いて、少しずつ正解に近づいていく反復解法（数値最適化）が **勾配降下法** です。

勾配 $\nabla J(\mathbf{w})$の方向は、パラメータ$\mathbf{w}$を変化させたときに $J(\mathbf{w})$ を最大に増加させる向きなので、$J(\mathbf{w})$を最大に減少させる逆方向へ少しだけ（学習率 $\eta$ 分だけ）パラメータを動かします。

$$\mathbf{w} \leftarrow \mathbf{w} - \eta \nabla J(\mathbf{w})$$

これを繰り返すことで、数値的に $\nabla J(\mathbf{w}^*)\simeq \mathbf{0}$ となる $\mathbf{w}^*$ を決定することができます。

<br>

#### **ロジスティック回帰モデルの勾配**
ロジスティック回帰モデルの場合、目的関数である交差エントロピー誤差の勾配は次のように「予測誤差 $\times$ 入力」の総和という綺麗な表式になります。

$$\nabla J(\mathbf{w}) = \sum_{i=1}^n (\underbrace{p_i - y_i}_{\text{予測誤差}}) \mathbf{x}_i= X^\top (\mathbf{p} - \mathbf{y})$$

ここで、全データをまとめた行列 $X$（$n$行$d$列）と、予測確率ベクトル $\mathbf{p}$、正解ベクトル $\mathbf{y}$ を使うと、非常にシンプルに記述できることがわかります。

$$\mathbf{w} \leftarrow \mathbf{w} - \eta X^\top (\mathbf{p} - \mathbf{y})$$

<br>

ロジスティック回帰モデルは、線形モデルの出力をシグモイド関数で確率に変換し、観測データの **「尤度の最大化」** によって最適な分類境界を決定するモデルです。また、この最適な境界を効率的に探索できる背景には、目的関数の勾配が解析的に（綺麗な数式で）得られるという性質が大きく関わっています。

## **Python 演習課題**

### **mva_06-A**

✅ **目的**: シグモイド関数の微分から勾配を数理的に導出し、その勾配式を用いてパラメータが最適化されるプロセス（勾配降下法）を視覚的に理解します。あわせて、シグモイド関数による確率曲線がデータに適合していく様子も可視化して確認しましょう。

#### 🖊 **【数理理解】勾配の導出と学習プロセス**

複雑に見える対数関数とシグモイド関数ですが、これらを組み合わせて微分すると、奇跡的に項が打ち消し合い（キャンセル）、劇的にシンプルな勾配式が得られます。
1変数（重み $w$, バイアス $b$; $z_i =w x_i+b$ ）のケースで、その導出過程を詳しく追ってみましょう。

##### **1. 準備：シグモイド関数の微分**
シグモイド関数 $\sigma(z) = \frac{1}{1+e^{-z}}$ を $z$ で微分すると、「自分自身 $\sigma(z) = p$ を使って書ける」 という非常に美しい性質があります。

$$\frac{\partial p}{\partial z} = \sigma'(z) = \left( \frac{1}{1+e^{-z}} \right)' = p (1 - p)$$


##### **2. 連鎖律による勾配の計算**

$$J(w,b) =- \sum_{i=1}^n \left[ y_i \log p_i + (1-y_i) \log(1-p_i) \right]=\sum_{i=1}^nJ_i$$

あるデータ1つ分に関する誤差 $J_i$ を、パラメータ $w$ で偏微分します。
これには「合成関数の微分（連鎖律）」を使います。

$$\frac{\partial J_i}{\partial w} = \underbrace{\frac{\partial J_i}{\partial p_i}}_{(A)} \cdot \underbrace{\frac{\partial p_i}{\partial z_i}}_{(B)} \cdot \underbrace{\frac{\partial z_i}{\partial w}}_{(C)}$$

それぞれ計算してみましょう。

- (A) 誤差関数の微分:
$$\frac{\partial J_i}{\partial p_i} = \frac{p_i - y_i}{p_i(1-p_i)}$$

- (B) シグモイドの微分:
$$\frac{\partial p_i}{\partial z_i} = p_i (1 - p_i)$$

- (C) 線形結合の微分: $z_i = w x_i + b$
$$\frac{\partial z_i}{\partial w} = x_i$$

##### **3. 最終的な勾配の導出**
これら(A)(B)(C)を掛け合わせると、見事に打ち消し合うことがわかります。

$$\frac{\partial J_i}{\partial w} = \frac{p_i - y_i}{p_i(1-p_i)} \cdot p_i(1-p_i) \cdot x_i = (p_i - y_i) x_i$$

これを全データ $n$ 個について足し合わせれば、全体の勾配式になります。

$$\frac{\partial J}{\partial w} = \sum_{i=1}^n (p_i - y_i) x_i$$

同様に、バイアス $b$ については $\frac{\partial z_i}{\partial b} = 1$ より以下の表式が得られます。

$$\frac{\partial J}{\partial b} = \sum_{i=1}^n (p_i - y_i)\cdot 1$$

この 「(予測 - 正解) $\times$ 入力」 という直感的な式が得られるからこそ、ロジスティック回帰は計算しやすく、広く使われているのです。

<br>

#### **具体例で確認してみよう！**

1変数のケース（勉強時間 $x$ から 合否 $y$を予測する）で導出した勾配式 $(p-y)x$ を使い、徐々に最適解へ近づいていく様子を確認してみましょう。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # 3Dプロット用

# 1. データの定義
# 勉強時間 (x) と 合否 (y: 0=不合格, 1=合格)
x = np.array([1, 2, 3, 5, 8, 9])
y = np.array([0, 0, 0, 1, 1, 1])

# --- 追加: データの確認 ---
# データをデータフレームにまとめて表示
df = pd.DataFrame({'Study Hours (x)': x, 'Pass (y)': y})

print("--- 使用するデータ ---")
try:
    display(df) # Jupyter Notebook / Colab 環境用
except NameError:
    print(df)   # 通常のPythonスクリプト環境用
# ------------------------

# 2. 必要な関数の定義
def sigmoid(z):
    """シグモイド関数: 入力を 0~1 の確率に変換"""
    return 1 / (1 + np.exp(-z))

def calculate_gradient(w, b, x, y):
    """
    現在のパラメータ (w, b) における勾配と損失を計算する関数
    """
    # (a) 予測 (線形結合 -> シグモイド)
    p = sigmoid(w * x + b)

    # (b) 勾配 (Gradient)
    # 解析的に導出したシンプルな式: sum( (p - y) * x )
    diff = p - y            # 予測と正解のズレ (誤差)
    dw = np.sum(diff * x)   # w の勾配 (誤差 × 入力)
    db = np.sum(diff)       # b の勾配 (x=1とみなす)

    # (c) 損失 (Cross-Entropy Error) も計算しておく（学習曲線の描画用）
    # 微小値 1e-7 を足して log(0) エラーを防ぐ
    loss = -np.sum(y * np.log(p + 1e-7) + (1 - y) * np.log(1 - p + 1e-7))

    return dw, db, loss, p

# --- 勾配降下法による学習ループ ---

# 初期パラメータ（わざと悪い値からスタート: ほぼ水平な線）
w, b = 0.1, 0.0
learning_rate = 0.2   # 学習率（大きくして矢印を見やすくする）
epochs = 1000         # 繰り返す回数

# 記録用リスト（軌跡の描画用）
history_loss = []
history_w = []
history_b = []

print(f"\nStart Training: w={w:.2f}, b={b:.2f}")

for i in range(epochs):
    # 現在の場所での勾配と損失を計算
    dw, db, loss, p = calculate_gradient(w, b, x, y)

    # 履歴に保存
    history_loss.append(loss)
    history_w.append(w)
    history_b.append(b)

    # パラメータの更新 (勾配の逆方向へ坂を下る)
    w = w - learning_rate * dw
    b = b - learning_rate * db

    if i % 100 == 0:
        print(f"Iter {i}: Loss={loss:.4f}, w={w:.2f}, b={b:.2f}")

print(f"Final: w={w:.2f}, b={b:.2f}, Loss={loss:.4f}")

# --- 可視化1: パラメータ空間での最適化の軌跡 (3D & 2D) ---

# 1. 損失関数の等高線・曲面を描くためのグリッドデータの作成
# 軌跡が含まれる範囲より少し広めに設定
w_min, w_max = min(history_w) - 1.0, max(history_w) + 1.0
b_min, b_max = min(history_b) - 1.0, max(history_b) + 1.0

# wとbの組み合わせを作成
ww, bb = np.meshgrid(np.linspace(w_min, w_max, 50),
                     np.linspace(b_min, b_max, 50))
Z_loss = np.zeros_like(ww)

# 各グリッド点での損失を計算
for i in range(ww.shape[0]):
    for j in range(ww.shape[1]):
        w_val, b_val = ww[i, j], bb[i, j]
        p_val = sigmoid(w_val * x + b_val)
        loss_val = -np.sum(y * np.log(p_val + 1e-7) + (1 - y) * np.log(1 - p_val + 1e-7))
        Z_loss[i, j] = loss_val

plt.figure(figsize=(16, 7))

# (1) 左側: 3D w-b-Loss 空間上の損失曲面と軌跡
ax1 = plt.subplot(1, 2, 1, projection='3d')

# 損失曲面の描画
ax1.plot_surface(ww, bb, Z_loss, cmap='viridis', alpha=0.5, edgecolor='none')

# 軌跡を3次元的にプロット (ボールが坂を転がり落ちるイメージ)
# 間引いてプロットして矢印を見やすくする
step = 5
ax1.plot(history_w[::step], history_b[::step], history_loss[::step], 'r.-', markersize=5, linewidth=1, label='Trajectory', zorder=10)
# スタートとゴール
ax1.scatter(history_w[0], history_b[0], history_loss[0], s=100, c='blue', marker='o', label='Start')
ax1.scatter(history_w[-1], history_b[-1], history_loss[-1], s=150, c='red', marker='*', label='End')

ax1.set_xlabel('Weight (w)')
ax1.set_ylabel('Bias (b)')
ax1.set_zlabel('Loss')
ax1.set_title('Loss Surface & Trajectory (3D)')
ax1.view_init(elev=30, azim=130) # 見やすい角度に調整
ax1.legend()

# (2) 右側: 2D w-b 平面上の等高線と軌跡
ax2 = plt.subplot(1, 2, 2)

# 背景に損失の等高線を描画
contour = ax2.contourf(ww, bb, Z_loss, levels=20, cmap='viridis', alpha=0.8)
plt.colorbar(contour, ax=ax2, label='Loss (Cross-Entropy)')

# パラメータ更新の軌跡をプロット
ax2.plot(history_w[::step], history_b[::step], 'r.-', markersize=5, linewidth=1, label='Trajectory')
# スタート地点とゴール地点
ax2.scatter(history_w[0], history_b[0], s=100, c='blue', marker='o', label='Start', edgecolors='white')
ax2.scatter(history_w[-1], history_b[-1], s=150, c='red', marker='*', label='End', edgecolors='white')

ax2.set_xlabel('Weight (w)')
ax2.set_ylabel('Bias (b)')
ax2.set_title('Gradient Descent Trajectory (2D)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

シグモイド関数による確率曲線がデータに適合していく様子も可視化して確認しましょう。

In [ ]:
# --- 可視化2: 学習経過によるシグモイド曲線の変化 ---
plt.figure(figsize=(10, 6))
x_range = np.linspace(0, 10, 100)

# プロットする反復回数（インデックス）を指定
# 初期変化が大きいため、0, 9, 99, 999 のように対数的な間隔で見ると変化がわかりやすい
step_indices = [0, 9, 99, 999]
num_plots = len(step_indices)

# 色の準備（青[初期] から 赤[最終] へ変化させる）
cmap = plt.get_cmap('coolwarm')
colors = cmap(np.linspace(0, 1, num_plots))

for i, idx in enumerate(step_indices):
    # インデックスが範囲外にならないよう調整
    if idx >= len(history_w):
        idx = len(history_w) - 1

    w_val = history_w[idx]
    b_val = history_b[idx]

    # 線のスタイル設定
    if i == 0:
        label = f'Start (Iter {idx})'
        linestyle = '--'
        alpha = 0.5
        width = 1.5
    elif i == num_plots - 1:
        label = f'Final (Iter {idx})'
        linestyle = '-'
        alpha = 1.0
        width = 3.0
    else:
        label = f'Iter {idx}'
        linestyle = '-'
        alpha = 0.6
        width = 1.5

    # プロット
    plt.plot(x_range, sigmoid(w_val * x_range + b_val),
             color=colors[i], linestyle=linestyle, linewidth=width, alpha=alpha,
             label=label)

# データ点
plt.scatter(x, y, c=y, cmap='bwr', s=100, edgecolors='k', label='Data', zorder=10)

plt.axhline(0.5, color='gray', linestyle=':', alpha=0.5)
plt.xlabel('Study Hours (x)')
plt.ylabel('Probability (p)')
plt.legend()
plt.title(f'Learning Process: Sigmoid Curve Fitting (Epochs={epochs})')
plt.grid(True, alpha=0.3)
plt.show()

### **mva_06-B**
✅ **目的**: ``scikit-learn`` ライブラリを用いてロジスティック回帰分析を効率的に実装し、確率の曲面（シグモイド曲面）がデータにどのようにフィットしているかを3次元的に可視化して確認しましょう。

#### 💻 **【実装・応用】 ``scikit-learn`` によるロジスティック回帰と確率曲面の可視化**
実務では、勾配降下法のループを自分で書くのではなく、最適化されたライブラリを使用します。詳細な仕様については、以下の公式ドキュメントを参照してください。

- ``scikit-learn`` 公式ドキュメント: ``LogisticRegression``

#### **具体例で確認してみよう！**
 ロジスティック回帰は、特徴空間において「0の平原」から「1の高原」へとつながる S字型の坂（シグモイド曲面） をデータに当てはめる作業とイメージできます。2変数のデータ（2科目の点数と合否）に対し、ライブラリを用いてモデルを構築します。さらに、``plotly`` を用いて、学習された確率の曲面を3次元的に可視化してみましょう。

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression

# 1. データの準備（2科目の点数と合否）
# Math(x1), English(x2) -> Pass(y)
data = {
    'Math':    [30, 40, 45, 50, 60, 65, 70, 80, 90, 95], # x1
    'English': [40, 30, 60, 50, 65, 80, 40, 85, 80, 90], # x2
    'Pass':    [ 0,  0,  0,  0,  0,  1,  1,  1,  1,  1]  # y
}
df = pd.DataFrame(data)
X = df[['Math', 'English']].values
y = df['Pass'].values

# --- 追加: データの確認 ---
print("--- 使用するデータ ---")
try:
    display(df) # Jupyter Notebook / Colab 環境用
except NameError:
    print(df)   # 通常のPythonスクリプト環境用
# ------------------------

# 2. ロジスティック回帰モデルの学習
# scikit-learnを使えば、最適化計算（勾配降下法など）は自動で行われます
model = LogisticRegression()
model.fit(X, y)

print("\n--- 学習されたパラメータ ---")
print(f"係数 (w1, w2): {model.coef_[0]}")
print(f"切片 (b): {model.intercept_[0]}")

# 3. 確率曲面（メッシュ）の作成
# x1, x2 の範囲でグリッドを作成
x1_min, x1_max = X[:, 0].min() - 5, X[:, 0].max() + 5
x2_min, x2_max = X[:, 1].min() - 5, X[:, 1].max() + 5
x1_range = np.linspace(x1_min, x1_max, 50)
x2_range = np.linspace(x2_min, x2_max, 50)
xx1, xx2 = np.meshgrid(x1_range, x2_range)

# グリッド上の全点について確率を予測
# predict_proba は [クラス0の確率, クラス1の確率] を返すので、[:, 1]でクラス1(合格)の確率を取得
Z_proba = model.predict_proba(np.c_[xx1.ravel(), xx2.ravel()])[:, 1]
Z_proba = Z_proba.reshape(xx1.shape)

# 4. Plotlyによる3D可視化
fig = go.Figure()

# (a) シグモイド曲面の描画
fig.add_trace(go.Surface(
    x=x1_range, y=x2_range, z=Z_proba,
    colorscale='RdBu', opacity=0.8,
    name='Probability Surface',
    colorbar=dict(title='Probability', x=1.05) # カラーバーを少し右へ
))

# (b) データ点の描画 (合格=1 は上空、不合格=0 は地面に配置)
# Pass (y=1) -> Red
fig.add_trace(go.Scatter3d(
    x=df[df['Pass']==1]['Math'],
    y=df[df['Pass']==1]['English'],
    z=df[df['Pass']==1]['Pass'],
    mode='markers', marker=dict(size=6, color='red', line=dict(color='black', width=1)),
    name='Pass (1)'
))
# Fail (y=0) -> Blue
fig.add_trace(go.Scatter3d(
    x=df[df['Pass']==0]['Math'],
    y=df[df['Pass']==0]['English'],
    z=df[df['Pass']==0]['Pass'],
    mode='markers', marker=dict(size=6, color='blue', line=dict(color='black', width=1)),
    name='Fail (0)'
))

# レイアウト設定
fig.update_layout(
    title='Logistic Regression: Sigmoid Surface Fitting (3D)',
    scene=dict(
        xaxis_title='Math Score',
        yaxis_title='English Score',
        zaxis_title='Probability (Pass)',
        zaxis=dict(range=[-0.1, 1.1]) # 確率なので0~1の範囲
    ),
    width=800, height=600,
    margin=dict(l=0, r=0, b=0, t=40),
    legend=dict(x=0.01, y=0.99) # 凡例を左上に配置してカラーバーとの重複を回避
)

fig.show()

$p=0.5$ に対応する決定境界がどのような曲線になっているか確認してみましょう。

In [ ]:
# 5. Matplotlibによる2D散布図と決定境界の可視化
plt.figure(figsize=(8, 6))

# 背景に確率の等高線 (Contour plot) を描画
contour = plt.contourf(xx1, xx2, Z_proba, alpha=0.8, cmap='RdBu_r', levels=np.linspace(0, 1, 11))
plt.colorbar(contour, label='Probability of Pass (Class 1)')

# 決定境界 (p=0.5) を白線で描画
plt.contour(xx1, xx2, Z_proba, levels=[0.5], colors='white', linewidths=2)

# 元のデータ点をプロット
plt.scatter(X[y==0, 0], X[y==0, 1], c='blue', edgecolors='k', label='Fail (0)', s=80)
plt.scatter(X[y==1, 0], X[y==1, 1], c='red', edgecolors='k', label='Pass (1)', s=80)

plt.xlabel('Math Score')
plt.ylabel('English Score')
plt.title('Logistic Regression: Decision Boundary (2D)')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.show()